# Feature Engineering

In [51]:
from preprocessing import get_data, join_and_sort, set_indexes, create_cleaned_df
from schema.schema import Venue
from main import engine
from sqlalchemy.sql import text
import pandas as pd
from tqdm import tqdm

In [52]:
tqdm.pandas()

In [53]:
from IPython.display import display_html
from itertools import chain,cycle
def display_side_by_side(*args,titles=cycle([''])):
    html_str=''
    for df,title in zip(args, chain(titles,cycle(['</br>'])) ):
        html_str+='<th style="text-align:center"><td style="vertical-align:top">'
        html_str+=f'<h2 style="text-align: center;">{title}</h2>'
        html_str+=df.to_html().replace('table','table style="display:inline"')
        html_str+='</td></th>'

    html_str = '<div style="display:flex;gap:10px;">' + html_str + '</div>'
    display_html(html_str,raw=True)

## Location

For now fill in with dummy values

- Distance between two teams
- Distance from equator (optional make up relevant landmarks given your dataset)
- Travel distance for away team

In [221]:
venues = pd.read_sql("SELECT * FROM venue", f"sqlite:///../db.sqlite3")

In [222]:
venues

,id,name,address,city,capacity,surface,image,team_id
0,494,Emirates Stadium,Queensland Road,London,60383,grass,https://media.api-sports.io/football/venues/49...,42
1,495,Villa Park,Trinity Road,Birmingham,42788,grass,https://media.api-sports.io/football/venues/49...,66
2,500,Ewood Park,Nuttall Street,"Blackburn, Lancashire",31367,grass,https://media.api-sports.io/football/venues/50...,67
3,504,Vitality Stadium,"Dean Court, Kings Park","Bournemouth, Dorset",12000,grass,https://media.api-sports.io/football/venues/50...,35
4,508,The American Express Community Stadium,Village Way,"Falmer, East Sussex",31872,grass,https://media.api-sports.io/football/venues/50...,51
5,512,Turf Moor,Harry Potts Way,Burnley,22546,grass,https://media.api-sports.io/football/venues/51...,44
6,516,Cardiff City Stadium,Leckwith Road,Caerdydd,33280,grass,https://media.api-sports.io/football/venues/51...,43
7,519,Stamford Bridge,Fulham Road,London,41841,grass,https://media.api-sports.io/football/venues/51...,49
8,525,Selhurst Park,Holmesdale Road,London,26309,grass,https://media.api-sports.io/football/venues/52...,52
9,527,Pride Park Stadium,"Pride Park, Royal Way",Derby,33597,grass,https://media.api-sports.io/football/venues/52...,69


In [54]:
# get venues
venues_statement = text("SELECT * FROM venue")
with engine.connect() as conn:
    venues = conn.execute(venues_statement)

venues_df = pd.DataFrame(venues)

In [55]:
# grabbing data
stats_df, fix_df = get_data()
set_indexes(stats_df, fix_df)
df = join_and_sort(stats_df, fix_df)
cleaned = create_cleaned_df(df)

In [56]:
venues_df.columns

Index(['id', 'name', 'address', 'city', 'capacity', 'surface', 'image',
       'team_id'],
      dtype='object')

In [57]:
# joining home team venue
to_remove = ["id", "surface", "image"]
cleaned = cleaned.reset_index().set_index('home_team_id').join(venues_df.set_index("team_id"))
cleaned.drop(to_remove, axis=1, inplace=True)

In [58]:
# joining away team venue
cleaned = cleaned.reset_index().set_index('away_team_id').join(venues_df.set_index("team_id"), rsuffix="_away")
cleaned.drop(to_remove, axis=1, inplace=True)

In [59]:
# distance traveled by away team
cleaned['address_away'].iloc[0], cleaned['address'].iloc[0]

('Rowsley Street', 'Fulham Road')

In [60]:
from geopy.geocoders import Nominatim
from geopy import distance

In [61]:
def append_uk(address: str):
    return address + ", UK"

In [62]:
geolocator = Nominatim(user_agent="soccer-ml")

In [63]:
# distance between two places
from geopy import distance

In [64]:
coordinates_cache = dict()
distance_cache = dict()
d_error_cache = dict()
c_error_cache = set()

In [65]:
errors = 0
successes = 0
used_cache = 0

In [66]:
# TODO: speed up IO bound function

In [67]:
def get_lat_long(row, col):
    global successes 
    global errors
    global used_cache
    suffix = ""
    if "away" in col:
        suffix = "_away"
    address, city = f"address{suffix}", f"city{suffix}" 
    # return cached result
    if row[col] in coordinates_cache:
        return coordinates_cache[row[col]]
    
    if row[col] in c_error_cache:
        return None
    
    # get lattitude and longitude locations
    location = None
    try:
        location = geolocator.geocode(f'{row[address]} {row[city]}, UK')
    except Exception:
        c_error_cache.add(row[col])
        return None

    if location:
        coordinates = (location.latitude, location.longitude)
        coordinates_cache[row[col]] = (location.latitude, location.longitude)
        return coordinates
    return None


def get_distance(row):
    """ get distance """
    if not row["home_coordinates"] or not row["away_coordinates"]:
        return None
    key = (row["home_team_id"], row["away_team_id"])
    if key in distance_cache:
        return distance_cache[key]

    d = distance.distance(row["home_coordinates"], row["away_coordinates"]).miles
    distance_cache[key] = d
    return d

In [68]:
cleaned["home_coordinates"] = cleaned.progress_apply(lambda row: get_lat_long(row, "home_team_id"), axis=1)

100%|██████████| 3042/3042 [06:59<00:00,  7.25it/s]


In [69]:
cleaned.reset_index(inplace=True)

In [70]:
cleaned["away_coordinates"] = cleaned.progress_apply(lambda row: get_lat_long(row, "away_team_id"), axis=1)

100%|██████████| 3042/3042 [02:34<00:00, 19.69it/s]


In [71]:
cleaned["distance_traveled"] = cleaned.progress_apply(get_distance, axis=1)

100%|██████████| 3042/3042 [00:00<00:00, 50977.56it/s]


## Continuous Features 
- Assist Distribution

In [72]:
# viewing feature importances
from joblib import load, dump
from sklearn.model_selection import train_test_split, GridSearchCV
import numpy as np

In [73]:
home_fi = load('../data/trial_2/features/home_features.joblib')
home_fi[:10]

[(0.18, 'numerical__home_team_id_assists_2'),
 (-0.113, 'datetime__start_time_weekday_thursday'),
 (0.108, 'numerical__home_team_id_assists'),
 (-0.092, 'numerical__A_GF_AH'),
 (-0.084, 'numerical__away_home_max_blocks'),
 (0.078, 'datetime__start_time_weekday_tuesday'),
 (-0.076, 'numerical__away_home_mean_interceptions'),
 (0.075, 'numerical__home_team_id_assists_3'),
 (0.071, 'numerical__home_team_id_goals_total'),
 (-0.068, 'numerical__away_team_id_goals_total_3')]

In [74]:
import plotly.express as px

In [75]:
fi_df = pd.DataFrame({"feature": [fi[1] for fi in home_fi[:10]], "|coefficient|": [np.abs(fi[0]) for fi in home_fi[:10]]})

In [76]:
fi_df

,feature,|coefficient|
0,numerical__home_team_id_assists_2,0.180
1,datetime__start_time_weekday_thursday,0.113
2,numerical__home_team_id_assists,0.108
3,numerical__A_GF_AH,0.092
4,numerical__away_home_max_blocks,0.084
5,datetime__start_time_weekday_tuesday,0.078
6,numerical__away_home_mean_interceptions,0.076
7,numerical__home_team_id_assists_3,0.075
8,numerical__home_team_id_goals_total,0.071
9,numerical__away_team_id_goals_total_3,0.068


In [77]:
fig = px.line(fi_df, x="feature", y="|coefficient|", title="Home Team Feature Importances")
fig.show()

## Assist Dist

### Small Sample Example

## Apply to our dataset

In [120]:
grouped = df.set_index(["fixture_id", "team_id", "substitute"])[["player_id", "assists", "goals_total"]].groupby(["player_id"], as_index=False)

In [121]:
sums = grouped.rolling(38, min_periods=1, closed='left').sum()
sums

,,,player_id,assists,goals_total
fixture_id,team_id,substitute,,,
79,36,0,0,NaN,NaN
868025,50,0,5,NaN,NaN
868033,50,0,5,NaN,NaN
868042,50,0,5,NaN,NaN
868051,50,0,5,NaN,NaN
...,...,...,...,...,...
868103,63,1,396757,NaN,NaN
868163,63,1,396757,NaN,NaN
868021,63,1,396757,NaN,NaN


In [122]:
cleaned

,away_team_id,home_team_id,fixture_id,team_id,player_id,minutes,rating,substitute,position,goals_total,...,address,city,capacity,name_away,address_away,city_away,capacity_away,home_coordinates,away_coordinates,distance_traveled
0,50,49,192904,50,105426,13.0,NaN,0,M,NaN,...,Fulham Road,London,41841,Etihad Stadium,Rowsley Street,Manchester,55097,"(51.4800296, -0.1950086)","(53.4829198, -2.2044599)",162.388310
1,66,42,192905,42,18769,70.0,NaN,0,M,1.0,...,Queensland Road,London,60383,Villa Park,Trinity Road,Birmingham,42788,"(51.55325205, -0.10809473400207634)","(52.5077598, -1.8834666)",100.430648
2,40,76,192966,40,105422,26.0,NaN,0,M,NaN,...,Normandy Road,Swansea,21028,Anfield,Anfield Road,Liverpool,55212,"(51.6413446, -3.9325347)","(53.43091645, -2.9609313526360888)",130.340815
3,47,33,192297,47,170,90.0,NaN,0,D,NaN,...,Sir Matt Busby Way,Manchester,76212,Tottenham Hotspur Stadium,"Bill Nicholson Way, 748 High Road",London,62850,"(53.4614265, -2.2881292)",None,NaN
4,746,46,192300,746,114744,90.0,NaN,0,M,NaN,...,Filbert Way,"Leicester, Leicestershire",34310,NaN,NaN,NaN,NaN,None,"(52.078191, 4.3083936)",NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3037,40,41,868325,41,303460,12.0,6.3,1,F,NaN,...,Britannia Road,"Southampton, Hampshire",32689,Anfield,Anfield Road,Liverpool,55212,None,"(53.43091645, -2.9609313526360888)",NaN
3038,51,66,868317,51,26475,89.0,7.2,0,F,1.0,...,Trinity Road,Birmingham,42788,The American Express Community Stadium,Village Way,"Falmer, East Sussex",31872,"(52.5077598, -1.8834666)","(50.8609642, -0.0794435)",137.729926
3039,50,55,868318,55,15745,1.0,NaN,1,D,NaN,...,"166 Lionel Rd N, Brentford","Brentford, Middlesex",17250,Etihad Stadium,Rowsley Street,Manchester,55097,None,"(53.4829198, -2.2044599)",NaN
3040,39,42,868316,42,284540,NaN,NaN,1,M,NaN,...,Queensland Road,London,60383,Molineux Stadium,Waterloo Road,"Wolverhampton, West Midlands",34624,"(51.55325205, -0.10809473400207634)","(52.5879438, -2.1323777)",112.048728


In [123]:
games = [868033, 868021, 868163, 868103]
teams = [50, 33, 63, 47]

In [130]:
# level 0: fixture_id, level 1: team_id, level 2: substitute
# for each fixture grab the home team nlargest 11
assist_dist = pd.Series(
    sums.xs(games[3]).xs(teams[3]).xs(0)['goals_total'].nlargest(11).sort_values(ascending=False)
)

In [131]:
assist_dist.index = [f"top_assister_{val}" for val in range(1, 12)] 

In [133]:
assist_dist.loc["assists_skew"] = assist_dist.skew()

In [137]:
pd.concat([assist_dist, assist_dist])

top_assister_1     27.000000
top_assister_2      9.000000
top_assister_3      6.000000
top_assister_4      3.000000
top_assister_5      2.000000
top_assister_6      2.000000
top_assister_7      2.000000
top_assister_8      1.000000
top_assister_9           NaN
top_assister_10          NaN
top_assister_11          NaN
assists_skew        2.355727
top_assister_1     27.000000
top_assister_2      9.000000
top_assister_3      6.000000
top_assister_4      3.000000
top_assister_5      2.000000
top_assister_6      2.000000
top_assister_7      2.000000
top_assister_8      1.000000
top_assister_9           NaN
top_assister_10          NaN
top_assister_11          NaN
assists_skew        2.355727
Name: goals_total, dtype: float64

In [214]:
def get_dist_features(row):
    # TODO extract func
    
    # home team
    fixture_id, home_team_id, away_team_id = row["fixture_id"], row["home_team_id"], row["away_team_id"]
    
    # goals
    goals_dist = pd.Series(sums.xs(fixture_id).xs(home_team_id).xs(0)['goals_total'].replace(np.nan, 0).nlargest(11).sort_values(ascending=False))
    goals_dist.index = [f"home_top_scorer_{val}" for val in range(1, 12)]
    goals_dist.loc["home_goals_dist_skewness"] = goals_dist.skew()
    goals_quant = goals_dist.quantile([0.25, 0.5, 0.75])
    goals_quant.index = [f"home_goals_{0.25 * i}_quant" for i in range(1, 4)]
    home_goals_features = pd.concat([goals_dist, goals_quant])
    
    # assists
    home_assists_dist = pd.Series(sums.xs(fixture_id).xs(home_team_id).xs(0)['assists'].replace(np.nan, 0).nlargest(11).sort_values(ascending=False))
    home_assists_dist.index = [f"home_top_assistor_{val}" for val in range(1, 12)]
    home_assists_dist.loc["home_assists_dist_skewness"] = home_assists_dist.skew()
    home_assist_quant = home_assists_dist.quantile([0.25, 0.5, 0.75])
    home_assist_quant.index = [f"home_assists_{0.25 * i}_quant" for i in range(1, 4)]
    home_assist_features = pd.concat([assist_dist, home_assist_quant])
    
    # away team
    # goals
    goals_dist = pd.Series(sums.xs(fixture_id).xs(away_team_id).xs(0)['goals_total'].replace(np.nan, 0).nlargest(11).sort_values(ascending=False))
    goals_dist.index = [f"away_top_scorer_{val}" for val in range(1, 12)]
    goals_dist.loc["away_goals_dist_skewness"] = goals_dist.skew()
    goals_quant = goals_dist.quantile([0.25, 0.5, 0.75])
    goals_quant.index = [f"away_goals_{0.25 * i}_quant" for i in range(1, 4)]
    away_goal_features = pd.concat([goals_dist, goals_quant])
    
    # assists
    away_assists_dist = pd.Series(sums.xs(fixture_id).xs(away_team_id).xs(0)['assists'].replace(np.nan, 0).nlargest(11).sort_values(ascending=False))
    away_assists_dist.index = [f"away_top_assistor_{val}" for val in range(1, 12)]
    away_assists_dist.loc["away_assists_dist_skewness"] = away_assists_dist.skew()
    away_assist_quant = away_assists_dist.quantile([0.25, 0.5, 0.75])
    away_assist_quant.index = [f"away_assists_{0.25 * i}_quant" for i in range(1, 4)]
    away_assist_features = pd.concat([away_assists_dist, away_assist_quant])

    features = pd.concat([home_goals_features, home_assist_features, away_goal_features, away_assist_features])
    # print(features)
    return features

In [215]:
res = cleaned.apply(get_dist_features, result_type='expand', axis=1)

In [219]:
pd.concat([cleaned, res], axis=1)

,away_team_id,home_team_id,fixture_id,team_id,player_id,minutes,rating,substitute,position,goals_total,...,away_top_assistor_6,away_top_assistor_7,away_top_assistor_8,away_top_assistor_9,away_top_assistor_10,away_top_assistor_11,away_assists_dist_skewness,away_assists_0.25_quant,away_assists_0.5_quant,away_assists_0.75_quant
0,50,49,192904,50,105426,13.0,NaN,0,M,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
1,66,42,192905,42,18769,70.0,NaN,0,M,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
2,40,76,192966,40,105422,26.0,NaN,0,M,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
3,47,33,192297,47,170,90.0,NaN,0,D,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
4,746,46,192300,746,114744,90.0,NaN,0,M,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3037,40,41,868325,41,303460,12.0,6.3,1,F,NaN,...,1.0,1.0,1.0,1.0,0.0,0.0,0.731016,0.932754,1.000000,7.000000
3038,51,66,868317,51,26475,89.0,7.2,0,F,1.0,...,1.0,1.0,1.0,0.0,0.0,0.0,2.503737,0.750000,1.500000,2.000000
3039,50,55,868318,55,15745,1.0,NaN,1,D,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,2.017657,0.000000,0.500000,1.254414
3040,39,42,868316,42,284540,NaN,NaN,1,M,NaN,...,1.0,1.0,0.0,0.0,0.0,0.0,1.507628,0.000000,1.000000,1.000000


In [ ]:
# Interactions


## Categorical
- One Hot Encoding
- Target Encoding

In [ ]:
from trial import create_pipeline

## Ordinal
- Ordinal Encoding
- One Hot Encoding
- Target Encoding

## Date Time

- is_holiday
- time_since_sunrise
- is_work_hours
- is_night
- is_morning

## Project Types:
- Tabular
- TSA
- NLP
- Computer Vision
- Audio
- Recommendation Systems

## Foundational Knowledge
- Databases
- Data Structures and Algorithms
- Systems Design and Scalability
- Statistics
- Linear Algebra
- Calculus